<a href="https://colab.research.google.com/github/DavidMo55/API_inmoviliaria/blob/main/reto_codigo_legado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Operación: código legado

OPERACIÓN: CÓDIGO LEGADO — Reto de 30 minutos
============================================
Eres el nuevo líder técnico de una fábrica. Un practicante dejó un script
que analiza lecturas de sensores recorriendo los mismos datos muchas veces.
Tu misión: implementar procesar_optimizado(), conservando los resultados.

USO DEL NOTEBOOK
Ejecuta las celdas en orden. Completa procesar_optimizado() antes de ejecutar
las celdas de comprobación y comparación; mientras esté pendiente, esas celdas
mostrarán NotImplementedError de forma intencional.
No necesitas instalar librerías. Si quieres usar NumPy o pandas, puedes
instalarlos en una celda con: %pip install numpy pandas

DATOS
Cada fila es (sensor_id, temperatura_en_decimas_C, vibracion, activo).
La temperatura es un entero: 815 representa 81.5 grados Celsius.
None indica una lectura ausente. Los IDs van de 0 a numero_sensores - 1.
Una lectura es válida si está activa y AMBAS mediciones están presentes.
Una lectura válida es una alerta si temperatura >= 800 O vibracion >= 450.

RESULTADO: lista de tuplas, ordenada por ID, incluidos sensores sin datos:
(sensor_id, numero_validas, promedio_temperatura_C, vibracion_maxima, alertas)
Sin lecturas válidas: (sensor_id, 0, None, None, 0).
El promedio usa TODAS las lecturas válidas; no redondees el resultado.
Una lectura que excede ambos umbrales cuenta como UNA alerta.

REGLAS
1. Edita solo procesar_optimizado() y añade imports/auxiliares si necesitas.
2. No cambies los datos, el legado ni el verificador; no modifiques la entrada.
3. No llames al legado desde tu solución ni reutilices resultados en caché.
4. Devuelve listas/tuplas y valores nativos de Python (convierte si usas NumPy).
5. Debe funcionar para otros tamaños, semillas, datos vacíos y casos límite.

PISTAS (sin solución)
- ¿Cuántas veces se recorre la tabla completa?
- Construye una condición de validez y otra de alerta.
- Piensa en operaciones sobre columnas y agregaciones por sensor.
- Para un promedio necesitas un conteo y una suma.
- Asegúrate de conservar sensores que no tengan ninguna lectura válida.

CONDICIÓN DE VICTORIA
Pasar los casos límite y la comparación con el legado; medir ambas versiones
con los mismos datos. Objetivo orientativo: acelerar al menos 5x con 200 000
filas y 100 sensores. La duración depende del equipo: no se promete 2 minutos.
Los conteos y máximos deben coincidir exactamente. El promedio se compara con
tolerancia 1e-12, porque el orden de operaciones puede cambiar en punto flotante.
Explica también por qué tu cálculo es matemáticamente equivalente; pasar los
casos de prueba es evidencia, pero no una demostración para todas las entradas.
La medición incluye las conversiones que realices dentro de tu función y
excluye generar datos, verificar resultados e imprimirlos.

## 1. Preparación y generación de datos

In [4]:
import math
import random
import statistics
import time
from types import SimpleNamespace

def generar_datos(filas, numero_sensores, semilla=42):
    rng = random.Random(semilla)
    datos = []
    for _ in range(filas):
        sensor = rng.randrange(numero_sensores)
        temperatura = None if rng.random() < 0.03 else rng.randint(-200, 1200)
        vibracion = None if rng.random() < 0.04 else rng.randint(0, 600)
        activo = rng.random() >= 0.10
        datos.append((sensor, temperatura, vibracion, activo))
    return datos

## 2. Código legado
Observa cuántos recorridos hace sobre los datos.

In [5]:
def procesar_legado(datos, numero_sensores):
    resultado = []
    for sensor in range(numero_sensores):
        temperaturas = []
        for fila in datos:
            if fila[0] == sensor:
                if fila[3] and fila[1] is not None and fila[2] is not None:
                    temperaturas.append(fila[1])
        vibraciones = []
        for fila in datos:
            if fila[0] == sensor:
                if fila[3] and fila[1] is not None and fila[2] is not None:
                    vibraciones.append(fila[2])
        alertas = 0
        for fila in datos:
            if fila[0] == sensor:
                if fila[3] and fila[1] is not None and fila[2] is not None:
                    if fila[1] >= 800 or fila[2] >= 450:
                        alertas += 1
        cantidad = len(temperaturas)
        promedio = sum(temperaturas) / (10 * cantidad) if cantidad else None
        maximo = max(vibraciones) if cantidad else None
        resultado.append((sensor, cantidad, promedio, maximo, alertas))
    return resultado

## 3. Tu solución
Edita esta función. Puedes agregar imports y funciones auxiliares. Vuelve a ejecutar esta celda después de cada cambio.

In [9]:
def procesar_optimizado(datos, numero_sensores):
    # Inicializamos acumuladores independientes para cada sensor de 0 a numero_sensores - 1
    conteos = [0] * numero_sensores
    sumas_temp = [0] * numero_sensores
    max_vibraciones = [None] * numero_sensores
    alertas = [0] * numero_sensores

    # Un solo recorrido sobre la lista de datos (O(N))
    for sensor, temp, vib, activo in datos:
        if 0 <= sensor < numero_sensores:
            if activo and temp is not None and vib is not None:
                conteos[sensor] += 1
                sumas_temp[sensor] += temp

                # Actualizar el máximo de vibración de manera eficiente
                v_max = max_vibraciones[sensor]
                if v_max is None or vib > v_max:
                    max_vibraciones[sensor] = vib

                # Una lectura válida es alerta si temperatura >= 800 o vibración >= 450
                if temp >= 800 or vib >= 450:
                    alertas[sensor] += 1

    # Construir el resultado final estructurado como se espera (O(S))
    resultado = []
    for sensor in range(numero_sensores):
        cantidad = conteos[sensor]
        if cantidad > 0:
            promedio = sumas_temp[sensor] / (10.0 * cantidad)
            maximo = max_vibraciones[sensor]
        else:
            promedio = None
            maximo = None
        resultado.append((sensor, cantidad, promedio, maximo, alertas[sensor]))

    return resultado

## 4. Verificación y medición
Ejecuta estas definiciones sin modificarlas.

In [6]:
def verificar(esperado, obtenido):
    assert type(obtenido) is list, "Devuelve una lista."
    assert len(esperado) == len(obtenido), "Faltan o sobran sensores."
    for referencia, actual in zip(esperado, obtenido):
        assert type(actual) is tuple and len(actual) == 5, "Cada fila debe ser una tupla de 5 valores."
        for columna, (a, b) in enumerate(zip(referencia, actual)):
            contexto = f"Sensor {referencia[0]}, columna {columna}: esperado {a}, recibido {b}"
            if a is None:
                assert b is None, contexto
            elif columna == 2:
                assert type(b) in (int, float), contexto
                assert math.isclose(a, b, rel_tol=1e-12, abs_tol=1e-12), contexto
            else:
                assert type(b) is int and a == b, contexto

def casos_limite(funcion):
    casos = [
        ([], 3, [(0, 0, None, None, 0), (1, 0, None, None, 0), (2, 0, None, None, 0)]),
        ([(0, 800, 450, True), (0, 799, 449, True), (0, 900, 500, False),
          (0, None, 500, True), (0, 900, None, True), (1, -100, 0, True)],
         3, [(0, 2, 79.95, 450, 1), (1, 1, -10.0, 0, 0), (2, 0, None, None, 0)]),
        ([(0, 800, 0, True), (0, 0, 450, True), (0, 900, 500, True)],
         1, [(0, 3, 1700 / 30, 500, 3)]),
        ([(0, None, None, True), (1, 100, 10, False)],
         2, [(0, 0, None, None, 0), (1, 0, None, None, 0)]),
    ]
    for datos, sensores, esperado in casos:
        copia = list(datos)
        verificar(esperado, funcion(datos, sensores))
        assert datos == copia, "Tu función modificó la entrada."

def medir(funcion, datos, sensores):
    inicio = time.perf_counter()
    resultado = funcion(datos, sensores)
    return resultado, time.perf_counter() - inicio

In [7]:
def ejecutar(modo='legado', filas=200000, sensores=100, semilla=42, repeticiones=3):
    if modo not in ('legado', 'comprobar', 'comparar'):
        raise ValueError('Modo desconocido.')
    if filas < 0 or sensores < 1 or repeticiones < 1:
        raise ValueError('Usa filas >= 0, sensores >= 1 y repeticiones >= 1.')
    if not __debug__:
        raise RuntimeError('El verificador necesita las aserciones activadas.')
    args = SimpleNamespace(modo=modo, filas=filas, sensores=sensores,
                           semilla=semilla, repeticiones=repeticiones)
    casos_limite(procesar_legado)
    if args.modo != 'legado':
        casos_limite(procesar_optimizado)
        for semilla in (0, 7, 123):
            muestra = generar_datos(500, 7, semilla)
            copia = list(muestra)
            verificar(procesar_legado(muestra, 7), procesar_optimizado(muestra, 7))
            assert muestra == copia, 'Tu función modificó la entrada.'
        print('Casos límite y tres muestras aleatorias: correctos.')
    datos = generar_datos(args.filas, args.sensores, args.semilla)
    print(f'Datos: {len(datos):,} lecturas, {args.sensores} sensores; semilla {args.semilla}.')
    if args.modo == 'legado':
        resultado, duracion = medir(procesar_legado, datos, args.sensores)
        print(f'Legado: {duracion:.6f} s. Primeros cinco sensores:')
        for fila in resultado[:5]:
            print(fila)
        print('Lee el enunciado al inicio del notebook e implementa procesar_optimizado().')
        return
    copia = list(datos)
    tiempos = {'legado': [], 'optimizado': []}
    repeticiones = args.repeticiones if args.modo == 'comparar' else 1
    # Calentamiento fuera del cronómetro para reducir el efecto del primer import.
    procesar_optimizado(datos[:100], args.sensores)
    for repeticion in range(repeticiones):
        funciones = [('legado', procesar_legado), ('optimizado', procesar_optimizado)]
        if repeticion % 2:
            funciones.reverse()
        resultados = {}
        for nombre, funcion in funciones:
            resultados[nombre], duracion = medir(funcion, datos, args.sensores)
            tiempos[nombre].append(duracion)
            assert datos == copia, 'Se modificó la entrada.'
        verificar(resultados['legado'], resultados['optimizado'])
    lento = statistics.median(tiempos['legado'])
    rapido = statistics.median(tiempos['optimizado'])
    print(f'Resultados equivalentes. Mediana de {repeticiones} ejecución(es):')
    print(f'Legado: {lento:.6f} s | Optimizado: {rapido:.6f} s')
    print(f'Aceleración: {lento / rapido:.2f}x' if rapido else 'Tiempo optimizado inferior a la resolución del reloj.')

## 5. Ejecutar el legado
Esta celda funciona antes de resolver el reto. Puedes ajustar el tamaño de los datos.

In [8]:
ejecutar(modo='legado', filas=200000, sensores=100)

Datos: 200,000 lecturas, 100 sensores; semilla 42.
Legado: 2.862747 s. Primeros cinco sensores:
(0, 1664, 49.46844951923077, 598, 771)
(1, 1667, 50.021595680863825, 600, 771)
(2, 1657, 50.92480386240193, 600, 760)
(3, 1692, 50.50082742316785, 600, 782)
(4, 1746, 50.16093928980527, 600, 791)
Lee el enunciado al inicio del notebook e implementa procesar_optimizado().


## 6. Comprobar tu solución
Ejecuta después de implementar tu función. Incluye casos límite y muestras aleatorias.

In [10]:
ejecutar(modo='comprobar', filas=10000, sensores=100)

Casos límite y tres muestras aleatorias: correctos.
Datos: 10,000 lecturas, 100 sensores; semilla 42.
Resultados equivalentes. Mediana de 1 ejecución(es):
Legado: 0.090462 s | Optimizado: 0.002413 s
Aceleración: 37.49x


## 7. Comparar tiempos
Objetivo orientativo: al menos 5x de aceleración. Se informa la mediana de tres ejecuciones.

In [11]:
ejecutar(modo='comparar', filas=200000, sensores=100, repeticiones=3)

Casos límite y tres muestras aleatorias: correctos.
Datos: 200,000 lecturas, 100 sensores; semilla 42.
Resultados equivalentes. Mediana de 3 ejecución(es):
Legado: 2.059244 s | Optimizado: 0.051797 s
Aceleración: 39.76x


## 8. Explica tus resultados

- ¿Qué provocaba la lentitud?
- ¿Qué cambiaste y por qué es matemáticamente equivalente?
- ¿Cuánto tardó cada versión y qué aceleración obtuviste?
- ¿Cómo cambia el rendimiento al aumentar las filas o los sensores?

Escribe aquí tus conclusiones.